# CYGNSS inundation visualization with vegetation data in a form of split map

Import the necessary libraries for working with Google Earth Engine (GEE) and geemap.

In [1]:
import ee
import geemap
import matplotlib as mpl
import colormaps as cmaps

SET PROJECT NAME, authenticate and set up your Earth Engine environment by creating a Google Cloud Project and enabling the Earth Engine API (https://www.google.com/url?q=https%3A%2F%2Fconsole.cloud.google.com%2Fapis%2Fapi%2Fearthengine.googleapis.com), if you haven’t already done so. Detailed setup instructions are available at https://developers.google.com/earth-engine/guides/access#a-role-in-a-cloud-project

In [2]:
project_name = 'YOUR_EE_PROJECTNAME'
ee.Authenticate()
ee.Initialize(project=project_name) 

Load the CYGNSS collection

In [3]:
cygnss_collection = ee.ImageCollection("projects/sat-io/open-datasets/CYGNSS/inundation")

Set up the functions to (i) remove NaN values from CYGNSS data and (ii) convert inundation data to fractional inundation

In [4]:
def mask_inundation(image):
    image = image.updateMask(image.neq(255))  # mask no-data values (255)
    return image.divide(100)  # scale to fractional values

Select the period of interest from the CYGNSS collections, apply a function to remove NaN values and convert inundation data to fractional inundation, and calculate the mean value over the selected period.

In [5]:
mean_inundation_maps = (
    cygnss_collection
    .filterDate(ee.Date('2025-03-01'), ee.Date('2025-03-14'))
    .select('inundation_3day')
    .map(mask_inundation)
    .mean()
)


Load vegetation data from MODIS, select the period of interest, and calculate the mean EVI value over the selected period.

In [6]:
modis_dataset = ee.ImageCollection('MODIS/061/MOD13Q1')

start_date = '2025-03-01'
end_date = '2025-03-14'
modis_filtered = modis_dataset.filterDate(start_date, end_date)

modis_evi = modis_filtered.select('EVI').map(
    lambda image: image.multiply(0.0001))
mean_evi = modis_evi.mean()


Set the visualization parameters (colors and value ranges) for the plotted bands. You can define your own colormap. By default, GEE accepts HEX or CSS color codes. Separate visualization parameters are defined for inundation data (nominal range: 0 to 1) and EVI data.

In [7]:
cmap_inundation = mpl.colormaps["gist_earth_r"].resampled(256) # palette (gist_earth) from matplotlib (https://matplotlib.org/stable/users/explain/colors/colormaps.html)
palette_inundation_hex = [mpl.colors.rgb2hex(cmap_inundation(i)) for i in range(256)]

inundation_vis = {
    'min': 0,
    'max': 1,
    'palette': palette_inundation_hex
}


#NDVI palette and visualisation params
cmap_evi = cmaps.NEO_modis_ndvi.resampled(256)
palette_evi_hex = [mpl.colors.rgb2hex(cmap_evi(i)) for i in range(256)]

evi_vis = {
    'min': 0,
    'max': 1,
    'palette': palette_evi_hex
}


Plot the region of interest with inundation and EVI data.

In [8]:
# Region of interest – South Sudan
roi = ee.Geometry.Rectangle([
    33, 8,   # west, south
    28, 13     # east, north
])

Map = geemap.Map() # generate map
Map.add_basemap('HYBRID') # add the 'HYBRID' layer containing satellite imagery
left_layer = geemap.ee_tile_layer(mean_inundation_maps, inundation_vis, "Fractional Inundation") # Create left layer of split map with inundation data
right_layer = geemap.ee_tile_layer(mean_evi, evi_vis, 'EVI') # Create right layer of split map with EVI data
Map.split_map(left_layer, right_layer, left_label='Inundation', right_label='Vegetation') # Add left and right layer to the map
Map.add_colorbar(inundation_vis, label="Inundation Fraction", orientation="vertical", layer_name="Inundation", position="bottomleft") # Add the colorbar for inundation data on the left.
Map.add_colorbar(evi_vis, label="EVI", orientation="vertical", layer_name="EVI", position="bottomright") # Add the colorbar for vegetation data on the right.
Map.center_object(roi, 7) # Set the initial view to the ROI
Map # Display the map

Map(center=[10.503000482394638, 30.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…